# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")


## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

**NOTE**: ONLY `Description` column will be used for searchable content

Let's look at an example document to see if everything worked as expected!

In [3]:
print(f'--> RAW content:\n{synthetic_usecase_data[0]}')
print(f'\n--> Page Content:\n\t{synthetic_usecase_data[0].page_content}')
print(f'\n--> Metadata:')
for k,v in synthetic_usecase_data[0].metadata.items():
    print(f'\t{k}:\t{v}')


--> RAW content:
page_content='A low-latency inference system for multimodal agents in autonomous systems.' metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}

--> Page Content:
	A low-latency inference system for multimodal agents in autonomous systems.

--> Metadata:
	source:	./data/Projects_with_Domains.csv
	row:	0
	Project Title:	InsightAI 1
	Project Domain:	Security
	Secondary Domain:	Finance / FinTech
	Description:	A low-latency inference system for multimodal agents in autonomous systems.
	Judge Comments:	Technically ambitious and well-executed.
	Score:	85
	Project Name:	Project Aurora
	Judge Score:	9.5


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [4]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [5]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [7]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [8]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

#NOTE ON RunnablePassthrough: When you pipe two dictionaries together where neither contains a Runnable object, Python just merges them as regular dictionaries instead of creating an LCEL chain!
# RunnablePassthrough is a Runnable object, so when you pipe it, LCEL recognizes the entire thing as a chain and creates a RunnableSequence!

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [9]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the project domains include Healthcare / MedTech, Security, Productivity Assistants, Creative / Design / Media, Developer Tools / DevEx, E‑commerce / Marketplaces, and Writing & Content. The most frequently occurring domain among these projects appears to be "Healthcare / MedTech," which is mentioned multiple times. Therefore, the most common project domain in the dataset is Healthcare / MedTech.'

#### Human Comment:
This seems incorrect following is the real count per project type, healthcare is not even in the top 3

As Primary Domain
|Domain|Count|
|---|---|
|E‑commerce / Marketplaces|5|
|Legal / Compliance|5|
|Finance / FinTech|5|

As Secondary Domain
|Domain|Count|
|---|---|
|QA / Testing / Validation|7|
|Finance / FinTech|7|
|Writing & Content|5|

In [10]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there were use cases related to security. One example is the project titled "MediMind," which is a security-focused project under the Security domain. It involves a medical imaging solution that aims to improve early diagnosis through vision transformers, indicating a focus on healthcare security applications.'

|Type|Domain|Count|
|---|---|---|
|Primary|Security|4|
|Secondary|Security|3|

Security Projects
|Project Title|Project Domain|Secondary Domain|Project Name|
|---|---|---|---|
|InsightAI 1|Security|Finance / FinTech|Project Aurora|
|WealthifyAI 3|Developer Tools / DevEx|Security|SynthMind|
|SecureNest 12|Security|Writing & Content|Neural Canvas|
|DocuCheck 14|Research & Knowledge|Security|DeepHarvest|
|MediMind 17|Security|Legal / Compliance|BioForge|
|Pathfinder 24|Healthcare / MedTech|Security|LatticeFlow|
|InsightAI 36|Security|Legal / Compliance|SoundScape|


In [11]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges generally had positive comments about the fintech-related projects, highlighting their strength, maturity, and real-world impact. For example:\n\n- One project was described as "A clever solution with measurable environmental benefit."\n- Another was noted for being a "Promising idea with robust experimental validation."\n- One project was praised as a "Technically ambitious and well-executed."\n- Another received positive feedback for having "Solid work with impressive real-world impact."\n\nOverall, the judges appreciated the technical quality, innovation, and practical significance of the fintech projects.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [12]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [13]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [14]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is not explicitly stated. However, from the sample entries, the project domains include "Productivity Assistants," "Legal / Compliance," "Data / Analytics," and "Healthcare / MedTech." Since only a small sample is available, I cannot determine the most common domain overall. If you have more data or a larger dataset, I can help analyze it further.'

#### Human Comment:
This retrieval performed better, However it is not from secondary domain, but primary

As Primary Domain
|Domain|Count|
|---|---|
|E‑commerce / Marketplaces|5|
|Legal / Compliance|5|
|Finance / FinTech|5|

As Secondary Domain
|Domain|Count|
|---|---|
|QA / Testing / Validation|7|
|Finance / FinTech|7|
|Writing & Content|5|

In [15]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any usecases specifically related to security.'

#### Human Comment:
This time it was not correct

|Type|Domain|Count|
|---|---|---|
|Primary|Security|4|
|Secondary|Security|3|

In [16]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges described the fintech project (which is associated with the secondary domain "Finance / FinTech" and the primary domain "Productivity Assistants") as "Technically ambitious and well-executed."'

#### Human Comment:
Incorrect, It's 12 projects (primary and secondary domain) with Judge comments

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

------------------------------------------------------------------------------------------------------------------------------------------------------------
##### ✅ Answer
Based on literature:
* **Theoretical example query**: "projects about API" 
* **Literature claim**: BM25 is better for finding exact keywords (like technical terms, acronyms, or specific phrases), while embeddings find conceptually similar content even with different words.

**DISCLAIMER**

However i ran multiple tests comparing embeddings vs bm25 and reality was different ...

* **Testing Results**: Theoretically, BM25 should excel when queries contain multiple rare, specific terms that must co-occur, such as "reinforcement learning energy efficiency data centers." However, testing revealed that modern embeddings (text-embedding-3-small) matched or exceeded BM25 performance in every case. This occurs because short, technical descriptions favor semantic understanding over term frequency, and modern embeddings handle technical terminology as well as keyword matching. The key insight: theoretical advantages don't always manifest in practice with modern tools and specific data characteristics.

------------------------------------------------------------------------------------------------------------------------------------------------------------

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [17]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [18]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [19]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, there are multiple project domains mentioned, including Healthcare / MedTech, Security, and Productivity Assistants. With only these samples, it appears that the Healthcare / MedTech domain is the most common among the projects listed.'

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific usecases related to security mentioned. The usecases described focus on federated learning for improving privacy in healthcare applications, but do not explicitly address security concerns.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects were positive. For the project titled "PlanPilot," which is in the Finance / FinTech domain, the judges described it as "A clever solution with measurable environmental benefit" and gave it a high score of 8.4.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [22]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [23]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [24]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain based on the provided data appears to be "Healthcare / MedTech," which is mentioned multiple times throughout the document.'

In [25]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security mentioned in the provided context. Specifically, one project titled "InsightAI 1" under the Project Domain "Security" focuses on a "low-latency inference system for multimodal agents in autonomous systems." Additionally, "SecureNest 12" is another project in the security domain, describing a "low-latency inference system for multimodal agents in autonomous systems" with solid supporting data and a forward-looking approach.'

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally had positive feedback about the fintech-related projects. They described these projects as clever, promising, solid, impressive, and ambitious. For example, one judge called a fintech project "a clever solution with measurable environmental benefit," while others noted the approach as promising with robust validation, technically mature, well-executed, and having good potential for commercialization. Overall, the judges recognized the fintech projects for their technical quality, scalability, and real-world impact.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

------------------------------------------------------------------------------------------------------------------------------------------------------------------
##### ✅ Answer
Generating multiple reformulations of a user query improves recall by using different word choices that match different documents. For example, asking "What did judges say about fintech projects?" could be rewritten as "What feedback did finance applications get?" or "How were financial tech projects evaluated?" Each version might find documents the others missed because documents use varying terminology. 

Off course more queries = more searches = higher cost BUT Less probablity of missing relevant information

------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

**IN SHORT**: Splits documents into small chunks for searching, but returns the full parent documents.

In [27]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [28]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [29]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [30]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [31]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [32]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain cannot be definitively determined from the few examples shown. The sample includes domains such as Security, Healthcare / MedTech, Productivity Assistants, and Creative / Design / Media, but there is not enough information to identify which is the most prevalent overall.'

In [33]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security mentioned. The projects primarily focus on federated learning to improve privacy in healthcare applications and other domains, but security as a distinct use case is not explicitly referenced.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Based on the provided context, the judges commented positively on the fintech projects. Specifically:\n\n- For SkyForge, the judges said it was "A clever solution with measurable environmental benefit."\n- For GreenPulse, they described it as "Technically ambitious and well-executed."\n- For WealthifyAI, the judge noted it as a "Comprehensive and technically mature approach."\n\nOverall, the judges appeared to recognize the projects as innovative, technically strong, and environmentally conscious.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [35]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [36]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [37]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Legal / Compliance," which is mentioned multiple times across different projects.'

In [38]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, there is a project titled "MediMind 17" in the Security domain, which involves a medical imaging solution aimed at improving early diagnosis through vision transformers.'

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had a range of opinions about the fintech projects. For example, regarding the project "Pathfinder 27," judges praised it for having excellent code quality and the use of open-source libraries, awarding it a high judge score of 9.8. Another project, "SecureNest 28," was considered conceptually strong but was noted to need more benchmarking results, with a judge score of 9.0. Overall, the judges appreciated the technical quality and innovative aspects of the fintech projects, though some noted areas for further development or validation.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [40]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [41]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [42]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [43]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [44]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [45]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Customer Support / Helpdesk," which is mentioned twice. Other domains like "Developer Tools / DevEx," "Legal / Compliance," and "Writing & Content" are also present but with fewer instances. Therefore, the most common project domain in this dataset is **Customer Support / Helpdesk**.'

In [46]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, one project titled "BioForge" is described as a medical imaging solution for early diagnosis, which is listed under the Security domain. Additionally, "InsightAI" is a low-latency inference system for autonomous systems, also categorized under Security.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had varying comments about the fintech projects. For example, one project, "WealthifyAI 16," was described as having a "comprehensive and technically mature approach," and "AutoMate 5" was noted as a "forward-looking idea with solid supporting data." The project "InsightAI 1" received high praise for being "technically ambitious and well-executed," with a judge score of 9.5. Overall, judges recognized the projects\' technical ambition, execution quality, and real-world impact, though some comments also pointed to areas for improvement, such as adding qualitative analysis or enhancing clarity in communication.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

----------------------------------------------------------------------------------------------------------------------------------------------------------
##### ✅ Answer

Short, repetitive sentences confuse the algorithm—it groups unrelated questions together because they sound similar ("How do I reset..." vs "How do I cancel..."), or splits related ones because there's not enough content to detect similarity.

We could make combinations of retrieval methods to adjust it, for example, combine semantic search (embeddings) with keyword search (like BM25).
When semantic chunking groups the wrong FAQs together because of similar wording, keyword search catches the actual topic differences—"password" vs "billing" vs "shipping"—and corrects the retrieval.

----------------------------------------------------------------------------------------------------------------------------------------------------------

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [49]:
# Load/Prepare the data
from langchain_community.document_loaders import PyMuPDFLoader
import pprint

loader = PyMuPDFLoader(file_path='data/howpeopleuseai.pdf')
pdf_docs = loader.load()

pprint.pp(pdf_docs[0].metadata)


{'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, '
             'AppendMode 1.1',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2025-09-12T20:05:32+00:00',
 'source': 'data/howpeopleuseai.pdf',
 'file_path': 'data/howpeopleuseai.pdf',
 'total_pages': 64,
 'format': 'PDF 1.6',
 'title': 'How People Use ChatGPT',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2025-09-15T10:32:36-04:00',
 'trapped': '',
 'modDate': "D:20250915103236-04'00'",
 'creationDate': 'D:20250912200532Z',
 'page': 0}


In [50]:
# Configure RAGAS System
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


For the next section, this is important for my personal understanding

The transformations act as a cleaning and structuring layer:

``` python
    Raw PDF → default_transforms → Clean, Structured KG → Test Generation ✓
            (chunks, extracts,
            normalizes, embeds)

    Raw PDF → (no transforms) → Test Generation ✗
                                (validation errors, bad extractions)
```

Specifically, transformations:

* Chunk intelligently - respecting document structure
* Clean text - remove artifacts, normalize formatting
* Extract properly - use LLM to identify real themes/entities (not dates!)
* Validate during transformation - catch issues early
* Create semantic relationships - so test generation understands context

In [56]:
from ragas.testset.graph import KnowledgeGraph, Node, NodeType

kg = KnowledgeGraph()

for doc in pdf_docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )

from ragas.testset.transforms import default_transforms, apply_transforms
import os

transformer_llm = generator_llm
embedding_model = generator_embeddings

# Apply transformations to enrich the knowledge graph
# WARNING: Only run apply_transforms() ONCE per knowledge graph!
# Running it multiple times will duplicate nodes/relationships and corrupt the KG.
# If you need to re-process:
#   - Either: Start fresh with a new KnowledgeGraph() object
#   - Or: Load the saved KG with KnowledgeGraph.load("knowledge_graph.json")

# Only transform if we don't already have an enriched KG
if not os.path.exists("knowledge_graph.json"):
    print("Creating transformations...")
    trans = default_transforms(documents=pdf_docs, llm=transformer_llm, embedding_model=embedding_model)
    apply_transforms(kg, trans)
    kg.save("knowledge_graph.json")
    print("Knowledge graph saved!")
    loaded_kg = kg  # Use the one we just created
else:
    print("knowledge_graph.json already exists. Loading instead of re-transforming...")
    loaded_kg = KnowledgeGraph.load("knowledge_graph.json")
    print("Knowledge graph loaded!")

loaded_kg

Creating transformations...


Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

...

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '2fd86f'. Skipping!
Property 'summary_embedding' already exists in node '795a39'. Skipping!
Property 'summary_embedding' already exists in node '0eaf71'. Skipping!
Property 'summary_embedding' already exists in node '627c4d'. Skipping!
Property 'summary_embedding' already exists in node 'd870ec'. Skipping!
Property 'summary_embedding' already exists in node '29b6ad'. Skipping!
Property 'summary_embedding' already exists in node '35fe11'. Skipping!
Property 'summary_embedding' already exists in node '8c576d'. Skipping!
Property 'summary_embedding' already exists in node 'd614cc'. Skipping!
Property 'summary_embedding' already exists in node 'f0a849'. Skipping!
Property 'summary_embedding' already exists in node '0249f0'. Skipping!
Property 'summary_embedding' already exists in node '4d4f66'. Skipping!
Property 'summary_embedding' already exists in node '3ce3c1'. Skipping!
Property 'summary_embedding' already exists in node '6f0c51'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Knowledge graph saved!


KnowledgeGraph(nodes: 86, relationships: 712)

In [57]:
# Create the golden dataset

from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings, knowledge_graph=loaded_kg)

dataset = generator.generate(testset_size=10, query_distribution=query_distribution)

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [58]:
#Save the dataset just in case

#dataset.to_pandas().to_json('test_dataset.json', orient='records')
dataset.to_pandas().to_csv('test_dataset.csv', index=False)

In [59]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"What does Bick et al., 2024, contribute to und...",[Introduction ChatGPT launched in November 202...,"Bick et al., 2024, study consumer usage of Cha...",single_hop_specifc_query_synthesizer
1,How does Claude compare to ChatGPT in terms of...,[Table 1: ChatGPT daily message counts (millio...,The context provides information about ChatGPT...,single_hop_specifc_query_synthesizer
2,What is Appendix D?,[Variation by Occupation Figure 23 presents va...,Appendix D contains a full report of GWA count...,single_hop_specifc_query_synthesizer
3,"Whaat does the 29,000 messages per second mean...",[Conclusion This paper studies the rapid growt...,"The 29,000 messages per second refers to the r...",single_hop_specifc_query_synthesizer
4,How does the use of privacy-preserving data ag...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context describes that ChatGPT employs a s...,multi_hop_abstract_query_synthesizer
5,How does the rapid growth and increasing adopt...,[<1-hop>\n\nConclusion This paper studies the ...,The rapid growth and widespread adoption of Ch...,multi_hop_abstract_query_synthesizer
6,Wht work-related message sharing and usage pat...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,"Based on the data, users in highly paid profes...",multi_hop_abstract_query_synthesizer
7,How does the growth of ChatGPT usage in the US...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"In the US, ChatGPT usage has grown rapidly, wi...",multi_hop_specific_query_synthesizer
8,"Based on the rapid growth of ChatGPT, which re...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT was used weekly by more ...",multi_hop_specific_query_synthesizer
9,"Hwo do Handa et al., 2025, and Handa et al. st...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"Based on the context, Handa et al., 2025, and ...",multi_hop_specific_query_synthesizer


#### Copy this Dataset to Langsmith to evaluate on Cost, Latency and Performance

In [60]:
# Enable Langchain tracing (langSmith)

os.environ["LANGCHAIN_TRACING_V2"] = "true"

from uuid import uuid4

# create langsmith project
os.environ["LANGCHAIN_PROJECT"] = f"AIM - S09-Assignment - {uuid4().hex[0:8]}"

In [61]:
# Create the DS on LangSmith and setting the Client

from langsmith import Client

client = Client()

dataset_name = "Synthetic Data for S09-Assignment"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="SD for Retrievers"
)

#Load questions to LangSmith

for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

#### Create VectorStore

In [62]:
# Create vector store

pdf_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

pdf_vectorstore = Qdrant.from_documents(
    documents=pdf_docs,
    embedding=pdf_embeddings,
    location=":memory:",
    collection_name="PDF_Synthetic_Questions"
)

#### Create RAG

In [63]:
# Retrievers

pdf_naive_retriever = pdf_vectorstore.as_retriever(search_kwargs={"k" : 10})
pdf_bm25_retriever = BM25Retriever.from_documents(pdf_docs)
pdf_compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=pdf_naive_retriever)
pdf_multi_query_retriever = MultiQueryRetriever.from_llm(retriever=pdf_naive_retriever, llm=chat_model) 

In [64]:
# PDF PARENT DOCUMENT RETRIEVER

pdf_parent_docs = pdf_docs
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

pdf_parent_client = QdrantClient(location=":memory:")

pdf_parent_client.create_collection(
    collection_name="pdf_full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

pdf_parent_document_vectorstore = QdrantVectorStore(
    collection_name="pdf_full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=pdf_parent_client
)

pdf_parent_store = InMemoryStore()

pdf_parent_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=pdf_parent_store,
    child_splitter=child_splitter,
)

pdf_parent_retriever.add_documents(pdf_parent_docs, ids=None)

In [65]:
# PDF ENSEMBLE RETRIEVER

pdf_retriever_list = [pdf_naive_retriever, pdf_bm25_retriever, pdf_compression_retriever, pdf_multi_query_retriever, pdf_parent_retriever]
equal_weighting = [1/len(pdf_retriever_list)] * len(pdf_retriever_list)

pdf_ensemble_retriever = EnsembleRetriever(retrievers=pdf_retriever_list, weights=equal_weighting)

In [66]:
# PDF SEMANTIC CHUNKING

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

pdf_semantic_documents = semantic_chunker.split_documents(pdf_docs)


pdf_semantic_vectorstore = Qdrant.from_documents(
    pdf_semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_PDF_Data_Semantic_Chunks"
)

pdf_semantic_retriever = pdf_semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

In [67]:
# LCEL RAG Chain Function to avoid repetition

def lcel_chain(retriever):
    return (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=itemgetter("context"))
        | {"output": rag_prompt | chat_model, "context": itemgetter("context")}
    )

In [68]:
pdf_naive_retriever_chain = lcel_chain(pdf_naive_retriever)
pdf_bm25_retriever_chain = lcel_chain(pdf_bm25_retriever)
pdf_compression_retriever_chain = lcel_chain(pdf_compression_retriever)
pdf_multi_query_retriever_chain = lcel_chain(pdf_multi_query_retriever)
pdf_parent_retriever_chain = lcel_chain(pdf_parent_retriever)
pdf_ensemble_retriever_chain = lcel_chain(pdf_ensemble_retriever)
pdf_semantic_retriever_chain = lcel_chain(pdf_semantic_retriever)

### Evaluation

In [69]:
# Capture results during LangSmith run

from langsmith.evaluation import evaluate as langsmith_evaluate
import time

# Storage for RAGAS
captured_results = {}

# Wrapper that captures results while LangSmith runs
class CapturingChain:
    def __init__(self, chain, storage_key, dataset_df, delay_seconds=0):
        self.chain = chain
        self.storage_key = storage_key
        self.dataset_df = dataset_df
        self.delay_seconds = delay_seconds
        self.call_count = 0
        captured_results[storage_key] = []
    
    def invoke(self, inputs):
        if self.call_count > 0 and self.delay_seconds > 0:
            time.sleep(self.delay_seconds)
        
        # Run the chain
        output = self.chain.invoke(inputs)
        
        # Capture for RAGAS
        question = inputs["question"]
        matching_row = self.dataset_df[self.dataset_df["user_input"] == question]
        if not matching_row.empty:
            captured_results[self.storage_key].append({
                "user_input": question,
                "response": output["output"].content,
                "retrieved_contexts": [doc.page_content for doc in output["context"]],
                "reference": matching_row.iloc[0]["reference"],
                "reference_contexts": matching_row.iloc[0]["reference_contexts"]
            })
        
        self.call_count += 1
        return output

In [75]:
# RAGAS evaluation setup with RETRIEVER-SPECIFIC metrics
from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness, 
    FactualCorrectness, 
    ResponseRelevancy, 
    ContextEntityRecall,
    ContextPrecision,      # ← Retriever: How relevant are retrieved chunks?
    ContextRecall,         # ← Retriever: Did we retrieve ground truth info?
)
from ragas import evaluate as ragas_evaluate
from ragas import RunConfig

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
custom_run_config = RunConfig(
    timeout=600,
    max_workers=4
)

# Evaluate
def evaluate_with_ragas(results, retriever_name):
    """Evaluate a retriever's results with RAGAS metrics"""
    print(f"Evaluating {retriever_name}...")
    
    # Convert to RAGAS dataset
    ragas_dataset = EvaluationDataset.from_list(results)
    
    # Evaluate with BOTH generation AND retriever metrics
    scores = ragas_evaluate(
        dataset=ragas_dataset,
        metrics=[
            # Retriever-specific metrics (main focus for assignment)
            ContextPrecision(),       # How precise is retrieval?
            ContextRecall(),          # How complete is retrieval?
            ContextEntityRecall(),    # Are key entities retrieved?
            # Generation metrics (show overall quality)
            Faithfulness(),           # Is response faithful to context?
            FactualCorrectness(),     # Is response factually correct?
            ResponseRelevancy(),      # Is response relevant to question?
        ],
        llm=evaluator_llm,
        run_config=custom_run_config
    )
    
    print(f"{retriever_name} completed!")
    return scores

In [71]:
# Run LangSmith evaluations (chains run once, results captured)
configs = [
    (pdf_naive_retriever_chain, "naive-retriever", "Naive Retriever", 0),
    (pdf_bm25_retriever_chain, "bm25-retriever", "BM25 Retriever", 0),
    (pdf_compression_retriever_chain, "compression-retriever", "Compression Retriever", 7),
    (pdf_multi_query_retriever_chain, "multiquery-retriever", "Multi-Query Retriever", 0),
    (pdf_parent_retriever_chain, "parent-retriever", "Parent Retriever", 0),
    (pdf_ensemble_retriever_chain, "ensemble-retriever", "Ensemble Retriever", 7),
    (pdf_semantic_retriever_chain, "semantic-retriever", "Semantic Chunk Retriever", 0),
]

langsmith_results = {}

for chain, exp_name, display_name, delay in configs:
    print(f"Running {display_name}...")
    capturing_chain = CapturingChain(chain, display_name, dataset.to_pandas(), delay)
    langsmith_results[exp_name] = langsmith_evaluate(
        capturing_chain.invoke,
        data="Synthetic Data for S09-Assignment",
        experiment_prefix=exp_name,
    )
    print(f"{display_name} completed!\n")

Running Naive Retriever...
View the evaluation results for experiment: 'naive-retriever-69c39c58' at:
https://smith.langchain.com/o/bb67da1a-f981-488f-9c09-ce7a028c911d/datasets/c4550807-137d-4058-862a-57e5dd0d1222/compare?selectedSessions=3d4ecb08-3a88-40d8-b5f3-1d163988a160




0it [00:00, ?it/s]

Naive Retriever completed!

Running BM25 Retriever...
View the evaluation results for experiment: 'bm25-retriever-62aeeafb' at:
https://smith.langchain.com/o/bb67da1a-f981-488f-9c09-ce7a028c911d/datasets/c4550807-137d-4058-862a-57e5dd0d1222/compare?selectedSessions=cacdfa36-9967-4635-aa91-11c307051c71




0it [00:00, ?it/s]

BM25 Retriever completed!

Running Compression Retriever...
View the evaluation results for experiment: 'compression-retriever-61355dc1' at:
https://smith.langchain.com/o/bb67da1a-f981-488f-9c09-ce7a028c911d/datasets/c4550807-137d-4058-862a-57e5dd0d1222/compare?selectedSessions=d0c0351c-2183-4644-92d7-51422e15f0ca




...

Parent Retriever completed!

Running Ensemble Retriever...
View the evaluation results for experiment: 'ensemble-retriever-7517d78f' at:
https://smith.langchain.com/o/bb67da1a-f981-488f-9c09-ce7a028c911d/datasets/c4550807-137d-4058-862a-57e5dd0d1222/compare?selectedSessions=4eaeec1c-6bd0-47f9-b228-068ed38dd907




0it [00:00, ?it/s]

Ensemble Retriever completed!

Running Semantic Chunk Retriever...
View the evaluation results for experiment: 'semantic-retriever-116ec442' at:
https://smith.langchain.com/o/bb67da1a-f981-488f-9c09-ce7a028c911d/datasets/c4550807-137d-4058-862a-57e5dd0d1222/compare?selectedSessions=38455fd9-fc79-4da2-8777-0f5ec4f905b5




0it [00:00, ?it/s]

Semantic Chunk Retriever completed!



In [76]:
# Run RAGAS with captured results
ragas_scores = {}
for display_name, results in captured_results.items():
    ragas_scores[display_name] = evaluate_with_ragas(results, display_name)

Evaluating Naive Retriever...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Naive Retriever completed!
Evaluating BM25 Retriever...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

BM25 Retriever completed!
Evaluating Compression Retriever...


...

Parent Retriever completed!
Evaluating Ensemble Retriever...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Ensemble Retriever completed!
Evaluating Semantic Chunk Retriever...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Semantic Chunk Retriever completed!


#### Display results

In [77]:
# TABLE 1: Aggregated Average Scores per Retriever (TRANSPOSED)
import pandas as pd

all_results = []
for name in ["Naive Retriever", "BM25 Retriever", "Compression Retriever", 
             "Multi-Query Retriever", "Parent Retriever", "Ensemble Retriever", 
             "Semantic Chunk Retriever"]:
    df = ragas_scores[name].to_pandas()
    df.insert(0, 'retriever', name)
    all_results.append(df)

results_comparison = pd.concat(all_results, ignore_index=True)

# Get metric columns (exclude metadata)
metric_columns = [col for col in results_comparison.columns 
                  if col not in ['retriever', 'user_input', 'retrieved_contexts', 
                                 'reference', 'reference_contexts', 'response']]

# Calculate mean scores per retriever
aggregated_scores = results_comparison.groupby('retriever')[metric_columns].mean()

# Reorder rows
aggregated_scores = aggregated_scores.reindex(["Naive Retriever", "BM25 Retriever", 
                                                "Compression Retriever", "Multi-Query Retriever", 
                                                "Parent Retriever", "Ensemble Retriever", 
                                                "Semantic Chunk Retriever"])

# TRANSPOSE: metrics as rows, retrievers as columns
aggregated_scores = aggregated_scores.T

print("📊 TABLE 1: Average Scores per Retriever")
print("=" * 120)
with pd.option_context('display.max_columns', None, 
                       'display.width', None,
                       'display.float_format', '{:.4f}'.format):
    display(aggregated_scores)

📊 TABLE 1: Average Scores per Retriever


retriever,Naive Retriever,BM25 Retriever,Compression Retriever,Multi-Query Retriever,Parent Retriever,Ensemble Retriever,Semantic Chunk Retriever
context_precision,0.9257,0.9000,1.0000,0.8470,1.0000,0.9468,0.9222
context_recall,0.8583,0.7583,0.8083,0.8917,0.6917,0.8583,0.8583
context_entity_recall,0.4276,0.4416,0.4312,0.3641,0.3397,0.4566,0.3817
faithfulness,0.9301,0.7615,0.8895,0.7758,0.8207,0.8362,0.8750
factual_correctness,0.5740,0.5680,0.6270,0.4930,0.5020,0.4940,0.5280
answer_relevancy,0.9455,0.7486,0.9311,0.9381,0.7493,0.8510,0.9386


In [88]:
from langsmith import Client
import pandas as pd

client = Client()

dataset_name = "Synthetic Data for S09-Assignment"
dataset = client.read_dataset(dataset_name=dataset_name)

# Session ID to Retriever Name mapping
session_to_retriever = {
    "38455fd9-fc79-4da2-8777-0f5ec4f905b5": "Semantic Chunk Retriever",
    "4eaeec1c-6bd0-47f9-b228-068ed38dd907": "Ensemble Retriever",
    "f0fc59fc-3d81-4b48-86ce-48015785bb41": "Parent Retriever",
    "52898192-9e53-4075-8554-bdf79381b950": "Multi-Query Retriever",
    "d0c0351c-2183-4644-92d7-51422e15f0ca": "Compression Retriever",
    "cacdfa36-9967-4635-aa91-11c307051c71": "BM25 Retriever",
    "3d4ecb08-3a88-40d8-b5f3-1d163988a160": "Naive Retriever"
}

# Get all examples
examples = list(client.list_examples(dataset_id=str(dataset.id)))

# Collect all run data
all_runs_data = []

for example in examples:
    runs = list(client.list_runs(reference_example_id=str(example.id)))
    
    for run in runs:
        # Convert session_id to string for lookup - THIS IS THE FIX
        session_id_str = str(run.session_id) if run.session_id else None
        retriever_name = session_to_retriever.get(session_id_str, "Unknown")
        
        duration = None
        if run.end_time and run.start_time:
            duration = (run.end_time - run.start_time).total_seconds()
        
        all_runs_data.append({
            'example_id': example.id,
            'run_id': run.id,
            'retriever': retriever_name,
            'session_id': session_id_str,
            'duration_seconds': duration,
            'total_cost': getattr(run, 'total_cost', None),
            'total_tokens': getattr(run, 'total_tokens', None),
        })

# Create DataFrame
df = pd.DataFrame(all_runs_data)

# print("=== Sample Data with Retriever Names ===")
# display(df.head(10))

# Calculate statistics by retriever
print("\n=== Cost & Latency Statistics ===")
stats = df.groupby('retriever').agg({
    'duration_seconds': 'mean',
    'total_cost': 'mean',
}).round(4)
stats.columns = ['Avg Latency (s)', 'Avg Cost ($)']
stats = stats.sort_values('Avg Cost ($)')
display(stats)


=== Cost & Latency Statistics ===


,Avg Latency (s),Avg Cost ($)
retriever,,
Parent Retriever,2.7436,0.000366
Compression Retriever,10.3713,0.000413
BM25 Retriever,3.3282,0.000458
Semantic Chunk Retriever,4.4336,0.000694
Naive Retriever,4.0653,0.000935
Multi-Query Retriever,6.1778,0.001195
Ensemble Retriever,14.4979,0.001358


# Conclussions

## Performance-Based Recommendation: Compression Retriever
* The **Compression Retriever** excels with perfect context precision (1.0) and the highest factual correctness (0.6270), ensuring retrieved information is both accurate and relevant. Combined with strong answer relevancy (0.9311), it delivers the most trustworthy responses.
* While the **Naive Retriever** edges ahead in faithfulness (0.9301 vs 0.8895) and answer relevancy (0.9455 vs 0.9311), Compression's superior factual correctness makes it the better choice when accuracy is paramount—despite its slower 10.37s latency.
## Cost & Latency-Based Recommendation: Parent Retriever
* For cost and latency optimization, the **Parent Retriever** is unmatched with the lowest cost ($0.000366) and fastest response time (2.74s)—making it 2.5x cheaper and 33% faster than the Naive Retriever.
* **BM25 Retriever** offers a reasonable alternative at $0.000458 and 3.33s.
* However, the **Parent Retriever's** poor performance scores (lowest context recall at 0.6917) mean cost savings come at a significant quality trade-off.


In [78]:
# TABLE 2: Raw scores by metric (one table per metric)

# Get all unique metrics
metric_columns = [col for col in results_comparison.columns 
                  if col not in ['retriever', 'user_input', 'retrieved_contexts', 
                                 'reference', 'reference_contexts', 'response']]

retriever_names = ["Naive Retriever", "BM25 Retriever", "Compression Retriever", 
                   "Multi-Query Retriever", "Parent Retriever", "Ensemble Retriever", 
                   "Semantic Chunk Retriever"]

for metric in metric_columns:
    print(f"\n{'='*120}")
    print(f"METRIC: {metric.upper()}")
    print(f"{'='*120}\n")
    
    # Start with metadata from first retriever
    first_retriever_data = results_comparison[results_comparison['retriever'] == retriever_names[0]]
    metric_table = first_retriever_data[['user_input', 'retrieved_contexts', 'response', 
                                          'reference', 'reference_contexts']].reset_index(drop=True)
    
    # Add scores from each retriever as columns
    for retriever_name in retriever_names:
        retriever_data = results_comparison[results_comparison['retriever'] == retriever_name]
        metric_table[retriever_name] = retriever_data[metric].reset_index(drop=True)
    
    # Display with better formatting
    with pd.option_context('display.max_rows', None,
                           'display.max_columns', None,
                           'display.width', None,
                           'display.max_colwidth', 50,  # Limit text column width
                           'display.float_format', '{:.4f}'.format):
        display(metric_table)


METRIC: CONTEXT_PRECISION



,user_input,retrieved_contexts,response,reference,reference_contexts,Naive Retriever,BM25 Retriever,Compression Retriever,Multi-Query Retriever,Parent Retriever,Ensemble Retriever,Semantic Chunk Retriever
0,"Hwo do Handa et al., 2025, and Handa et al. st...",[6\nWho Uses ChatGPT\nIn this section we repor...,Based on the studies by Handa et al. (2025) an...,"Based on the context, Handa et al., 2025, and ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.9627,1.0000,1.0000,0.3642,1.0000,0.9809,0.9889
1,"Based on the rapid growth of ChatGPT, which re...",[7\nConclusion\nThis paper studies the rapid g...,The data shows that since ChatGPT's rapid adop...,"By July 2025, ChatGPT was used weekly by more ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,How does the growth of ChatGPT usage in the US...,[Figure 5: Daily messages sent per weekly acti...,The growth of ChatGPT usage in the US is close...,"In the US, ChatGPT usage has grown rapidly, wi...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,1.0000,1.0000,1.0000,0.9787,1.0000,0.9914,1.0000
3,Wht work-related message sharing and usage pat...,[6.5\nVariation by Occupation\nFigure 23 prese...,The work-related message sharing and usage pat...,"Based on the data, users in highly paid profes...",[<1-hop>\n\nVariation by Occupation Figure 23 ...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
4,How does the rapid growth and increasing adopt...,[7\nConclusion\nThis paper studies the rapid g...,The rapid growth and increasing adoption of Ch...,The rapid growth and widespread adoption of Ch...,[<1-hop>\n\nConclusion This paper studies the ...,0.9750,1.0000,1.0000,0.9705,1.0000,0.9798,0.9861
5,How does the use of privacy-preserving data ag...,[Figure 2: Illustration of Aggregated Employme...,The use of privacy-preserving data aggregation...,The context describes that ChatGPT employs a s...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.8951,1.0000,1.0000,0.8813,1.0000,0.9024,0.8593
6,"Whaat does the 29,000 messages per second mean...",[1\nIntroduction\nChatGPT launched in November...,"The figure of 29,000 messages per second in th...","The 29,000 messages per second refers to the r...",[Conclusion This paper studies the rapid growt...,0.8529,1.0000,1.0000,0.8711,1.0000,1.0000,0.8611
7,What is Appendix D?,[D\nAppendix: Occupational Results\nD.0.1\nGWA...,"Appendix D is titled ""Appendix: Occupational R...",Appendix D contains a full report of GWA count...,[Variation by Occupation Figure 23 presents va...,0.6250,0.0000,1.0000,0.4769,1.0000,0.6222,0.5694
8,How does Claude compare to ChatGPT in terms of...,[Figure 6: The solid black line represents the...,"Based on the provided data, Claude and ChatGPT...",The context provides information about ChatGPT...,[Table 1: ChatGPT daily message counts (millio...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
9,"What does Bick et al., 2024, contribute to und...","[How People Use ChatGPT\nAaron Chatterji, Thom...","Bick et al., 2024, contribute to understanding...","Bick et al., 2024, study consumer usage of Cha...",[Introduction ChatGPT launched in November 202...,0.9468,1.0000,1.0000,0.9273,1.0000,0.9914,0.9571



METRIC: CONTEXT_RECALL



,user_input,retrieved_contexts,response,reference,reference_contexts,Naive Retriever,BM25 Retriever,Compression Retriever,Multi-Query Retriever,Parent Retriever,Ensemble Retriever,Semantic Chunk Retriever
0,"Hwo do Handa et al., 2025, and Handa et al. st...",[6\nWho Uses ChatGPT\nIn this section we repor...,Based on the studies by Handa et al. (2025) an...,"Based on the context, Handa et al., 2025, and ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.6667,0.6667,0.6667,1.0000,0.0000,0.6667,0.6667
1,"Based on the rapid growth of ChatGPT, which re...",[7\nConclusion\nThis paper studies the rapid g...,The data shows that since ChatGPT's rapid adop...,"By July 2025, ChatGPT was used weekly by more ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,How does the growth of ChatGPT usage in the US...,[Figure 5: Daily messages sent per weekly acti...,The growth of ChatGPT usage in the US is close...,"In the US, ChatGPT usage has grown rapidly, wi...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,0.6667,0.6667,0.6667,0.6667,0.6667,0.6667,0.6667
3,Wht work-related message sharing and usage pat...,[6.5\nVariation by Occupation\nFigure 23 prese...,The work-related message sharing and usage pat...,"Based on the data, users in highly paid profes...",[<1-hop>\n\nVariation by Occupation Figure 23 ...,1.0000,1.0000,1.0000,1.0000,0.7500,1.0000,1.0000
4,How does the rapid growth and increasing adopt...,[7\nConclusion\nThis paper studies the rapid g...,The rapid growth and increasing adoption of Ch...,The rapid growth and widespread adoption of Ch...,[<1-hop>\n\nConclusion This paper studies the ...,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000,0.5000
5,How does the use of privacy-preserving data ag...,[Figure 2: Illustration of Aggregated Employme...,The use of privacy-preserving data aggregation...,The context describes that ChatGPT employs a s...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,1.0000,1.0000,0.5000,1.0000,1.0000,1.0000,1.0000
6,"Whaat does the 29,000 messages per second mean...",[1\nIntroduction\nChatGPT launched in November...,"The figure of 29,000 messages per second in th...","The 29,000 messages per second refers to the r...",[Conclusion This paper studies the rapid growt...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
7,What is Appendix D?,[D\nAppendix: Occupational Results\nD.0.1\nGWA...,"Appendix D is titled ""Appendix: Occupational R...",Appendix D contains a full report of GWA count...,[Variation by Occupation Figure 23 presents va...,1.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000
8,How does Claude compare to ChatGPT in terms of...,[Figure 6: The solid black line represents the...,"Based on the provided data, Claude and ChatGPT...",The context provides information about ChatGPT...,[Table 1: ChatGPT daily message counts (millio...,0.7500,0.7500,0.7500,0.7500,0.0000,0.7500,0.7500
9,"What does Bick et al., 2024, contribute to und...","[How People Use ChatGPT\nAaron Chatterji, Thom...","Bick et al., 2024, contribute to understanding...","Bick et al., 2024, study consumer usage of Cha...",[Introduction ChatGPT launched in November 202...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000



METRIC: CONTEXT_ENTITY_RECALL



...

,user_input,retrieved_contexts,response,reference,reference_contexts,Naive Retriever,BM25 Retriever,Compression Retriever,Multi-Query Retriever,Parent Retriever,Ensemble Retriever,Semantic Chunk Retriever
0,"Hwo do Handa et al., 2025, and Handa et al. st...",[6\nWho Uses ChatGPT\nIn this section we repor...,Based on the studies by Handa et al. (2025) an...,"Based on the context, Handa et al., 2025, and ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,1.0000,0.5172,1.0000,1.0000,0.8571,0.9512,1.0000
1,"Based on the rapid growth of ChatGPT, which re...",[7\nConclusion\nThis paper studies the rapid g...,The data shows that since ChatGPT's rapid adop...,"By July 2025, ChatGPT was used weekly by more ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,1.0000,1.0000,0.9286,1.0000,0.4324,0.9730,0.9245
2,How does the growth of ChatGPT usage in the US...,[Figure 5: Daily messages sent per weekly acti...,The growth of ChatGPT usage in the US is close...,"In the US, ChatGPT usage has grown rapidly, wi...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,0.9474,0.9474,1.0000,1.0000,0.5385,0.8947,0.7778
3,Wht work-related message sharing and usage pat...,[6.5\nVariation by Occupation\nFigure 23 prese...,The work-related message sharing and usage pat...,"Based on the data, users in highly paid profes...",[<1-hop>\n\nVariation by Occupation Figure 23 ...,0.8889,1.0000,1.0000,1.0000,1.0000,1.0000,0.9722
4,How does the rapid growth and increasing adopt...,[7\nConclusion\nThis paper studies the rapid g...,The rapid growth and increasing adoption of Ch...,The rapid growth and widespread adoption of Ch...,[<1-hop>\n\nConclusion This paper studies the ...,1.0000,0.8696,0.9375,1.0000,0.5714,0.6071,1.0000
5,How does the use of privacy-preserving data ag...,[Figure 2: Illustration of Aggregated Employme...,The use of privacy-preserving data aggregation...,The context describes that ChatGPT employs a s...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.7742,0.5652,0.7857,1.0000,0.8077,0.7931,0.8500
6,"Whaat does the 29,000 messages per second mean...",[1\nIntroduction\nChatGPT launched in November...,"The figure of 29,000 messages per second in th...","The 29,000 messages per second refers to the r...",[Conclusion This paper studies the rapid growt...,0.8333,1.0000,1.0000,0.6667,1.0000,1.0000,1.0000
7,What is Appendix D?,[D\nAppendix: Occupational Results\nD.0.1\nGWA...,"Appendix D is titled ""Appendix: Occupational R...",Appendix D contains a full report of GWA count...,[Variation by Occupation Figure 23 presents va...,0.8889,0.0000,0.4444,0.5000,1.0000,0.5000,0.8000
8,How does Claude compare to ChatGPT in terms of...,[Figure 6: The solid black line represents the...,"Based on the provided data, Claude and ChatGPT...",The context provides information about ChatGPT...,[Table 1: ChatGPT daily message counts (millio...,0.9688,0.8158,0.9655,0.5909,1.0000,0.6429,0.9250
9,"What does Bick et al., 2024, contribute to und...","[How People Use ChatGPT\nAaron Chatterji, Thom...","Bick et al., 2024, contribute to understanding...","Bick et al., 2024, study consumer usage of Cha...",[Introduction ChatGPT launched in November 202...,1.0000,0.9000,0.8333,0.0000,1.0000,1.0000,0.5000



METRIC: FACTUAL_CORRECTNESS



,user_input,retrieved_contexts,response,reference,reference_contexts,Naive Retriever,BM25 Retriever,Compression Retriever,Multi-Query Retriever,Parent Retriever,Ensemble Retriever,Semantic Chunk Retriever
0,"Hwo do Handa et al., 2025, and Handa et al. st...",[6\nWho Uses ChatGPT\nIn this section we repor...,Based on the studies by Handa et al. (2025) an...,"Based on the context, Handa et al., 2025, and ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.3100,0.5000,0.5700,0.1000,0.0000,0.0000,0.4300
1,"Based on the rapid growth of ChatGPT, which re...",[7\nConclusion\nThis paper studies the rapid g...,The data shows that since ChatGPT's rapid adop...,"By July 2025, ChatGPT was used weekly by more ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.7700,0.7600,0.6900,0.6200,0.8000,0.8600,0.8000
2,How does the growth of ChatGPT usage in the US...,[Figure 5: Daily messages sent per weekly acti...,The growth of ChatGPT usage in the US is close...,"In the US, ChatGPT usage has grown rapidly, wi...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,0.7000,0.6400,0.7500,0.5600,0.7600,0.5300,0.6300
3,Wht work-related message sharing and usage pat...,[6.5\nVariation by Occupation\nFigure 23 prese...,The work-related message sharing and usage pat...,"Based on the data, users in highly paid profes...",[<1-hop>\n\nVariation by Occupation Figure 23 ...,0.7100,0.7600,0.8100,0.8100,0.7600,0.4500,0.7700
4,How does the rapid growth and increasing adopt...,[7\nConclusion\nThis paper studies the rapid g...,The rapid growth and increasing adoption of Ch...,The rapid growth and widespread adoption of Ch...,[<1-hop>\n\nConclusion This paper studies the ...,0.5700,0.4000,0.6200,0.6000,0.7100,0.4500,0.4000
5,How does the use of privacy-preserving data ag...,[Figure 2: Illustration of Aggregated Employme...,The use of privacy-preserving data aggregation...,The context describes that ChatGPT employs a s...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.6900,0.6700,0.7600,0.6200,0.7100,0.5200,0.6000
6,"Whaat does the 29,000 messages per second mean...",[1\nIntroduction\nChatGPT launched in November...,"The figure of 29,000 messages per second in th...","The 29,000 messages per second refers to the r...",[Conclusion This paper studies the rapid growt...,0.7500,0.6700,0.6000,0.4400,0.6700,0.6700,0.5700
7,What is Appendix D?,[D\nAppendix: Occupational Results\nD.0.1\nGWA...,"Appendix D is titled ""Appendix: Occupational R...",Appendix D contains a full report of GWA count...,[Variation by Occupation Figure 23 presents va...,0.4600,0.0000,0.4000,0.3300,0.2900,0.5300,0.2900
8,How does Claude compare to ChatGPT in terms of...,[Figure 6: The solid black line represents the...,"Based on the provided data, Claude and ChatGPT...",The context provides information about ChatGPT...,[Table 1: ChatGPT daily message counts (millio...,0.6700,0.5800,0.6300,0.6700,0.3200,0.7300,0.6700
9,"What does Bick et al., 2024, contribute to und...","[How People Use ChatGPT\nAaron Chatterji, Thom...","Bick et al., 2024, contribute to understanding...","Bick et al., 2024, study consumer usage of Cha...",[Introduction ChatGPT launched in November 202...,0.1100,0.7000,0.4400,0.1800,0.0000,0.2000,0.1200



METRIC: ANSWER_RELEVANCY



,user_input,retrieved_contexts,response,reference,reference_contexts,Naive Retriever,BM25 Retriever,Compression Retriever,Multi-Query Retriever,Parent Retriever,Ensemble Retriever,Semantic Chunk Retriever
0,"Hwo do Handa et al., 2025, and Handa et al. st...",[6\nWho Uses ChatGPT\nIn this section we repor...,Based on the studies by Handa et al. (2025) an...,"Based on the context, Handa et al., 2025, and ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.9207,0.9507,0.9103,0.9528,0.9205,0.9234,0.9557
1,"Based on the rapid growth of ChatGPT, which re...",[7\nConclusion\nThis paper studies the rapid g...,The data shows that since ChatGPT's rapid adop...,"By July 2025, ChatGPT was used weekly by more ...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.8993,0.8932,0.8985,0.9030,0.9244,0.8948,0.9243
2,How does the growth of ChatGPT usage in the US...,[Figure 5: Daily messages sent per weekly acti...,The growth of ChatGPT usage in the US is close...,"In the US, ChatGPT usage has grown rapidly, wi...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,0.9504,0.9048,0.9048,0.9193,0.9044,0.8861,0.9543
3,Wht work-related message sharing and usage pat...,[6.5\nVariation by Occupation\nFigure 23 prese...,The work-related message sharing and usage pat...,"Based on the data, users in highly paid profes...",[<1-hop>\n\nVariation by Occupation Figure 23 ...,0.9591,0.9567,0.9399,0.9567,0.9567,0.9567,0.9467
4,How does the rapid growth and increasing adopt...,[7\nConclusion\nThis paper studies the rapid g...,The rapid growth and increasing adoption of Ch...,The rapid growth and widespread adoption of Ch...,[<1-hop>\n\nConclusion This paper studies the ...,0.9221,0.9221,0.9221,0.9221,0.9443,0.9930,0.9147
5,How does the use of privacy-preserving data ag...,[Figure 2: Illustration of Aggregated Employme...,The use of privacy-preserving data aggregation...,The context describes that ChatGPT employs a s...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,0.9120,0.9251,0.9604,0.9352,0.9120,0.9173,0.9186
6,"Whaat does the 29,000 messages per second mean...",[1\nIntroduction\nChatGPT launched in November...,"The figure of 29,000 messages per second in th...","The 29,000 messages per second refers to the r...",[Conclusion This paper studies the rapid growt...,0.9766,0.9595,0.9663,0.9769,0.9672,0.9672,0.9761
7,What is Appendix D?,[D\nAppendix: Occupational Results\nD.0.1\nGWA...,"Appendix D is titled ""Appendix: Occupational R...",Appendix D contains a full report of GWA count...,[Variation by Occupation Figure 23 presents va...,0.9634,0.0000,0.8473,0.8632,0.9634,0.0000,0.8739
8,How does Claude compare to ChatGPT in terms of...,[Figure 6: The solid black line represents the...,"Based on the provided data, Claude and ChatGPT...",The context provides information about ChatGPT...,[Table 1: ChatGPT daily message counts (millio...,0.9670,0.0000,0.9631,0.9732,0.0000,0.9732,0.9802
9,"What does Bick et al., 2024, contribute to und...","[How People Use ChatGPT\nAaron Chatterji, Thom...","Bick et al., 2024, contribute to understanding...","Bick et al., 2024, study consumer usage of Cha...",[Introduction ChatGPT launched in November 202...,0.9848,0.9739,0.9981,0.9782,0.0000,0.9981,0.9415
